# Benchmarks - Rust

The one Rust example from [docs/benchmarks.md](https://platob.github.io/yggdryl/benchmarks/), in page order.

Generated by `scripts/build_docs_notebooks.py` from the blocks that
`scripts/check_docs_examples.py` compiles and runs, so every code cell below is
an example that passed. An edit here lives until the next build overwrites it.

The cells are unexecuted and expect the
[evcxr](https://github.com/evcxr/evcxr) kernel. Declare the crate once, in
a cell of your own, before running them:

```rust
:dep yggdryl = { version = "0.1", features = ["parquet", "iceberg"] }
```

### The pipeline those numbers measure

In [ ]:
use arrow_array::{Array, Int32Array, Int64Array, StringArray};
use yggdryl::io::IOBase;
use yggdryl::local::Folder;
use yggdryl::text::TextLineOptions;

let pattern = r"^\d{4}-\d{2}-\d{2} \d{2}:\d{2}:\d{2}\S* \[(?<level>[^\]]+)\] \[(?<logger>[^\]]+)\] \[(?<thread_id>\d+)\] took=(?<latency_us>\d+)";

let root = std::env::temp_dir().join("yggdryl-docs-gzip-lines");
let _ = std::fs::remove_dir_all(&root);
std::fs::create_dir_all(&root)?;

// Two rotated leaves; the second record of the first spans a stack trace.
let leaves = [
    concat!(
        "2024-02-01 10:00:00.000000 [ii] [engine] [3] took=120 fill 100 SYMB-0001\n",
        "2024-02-01 10:00:01.000000 [ee] [engine] [4] took=980 fill 101 SYMB-0002\n",
        "    at engine::match(order.rs:118)\n",
        "    at engine::step(order.rs:64)\n",
        "2024-02-01 10:00:02.000000 [ww] [router] [5] took=240 fill 102 SYMB-0003\n",
    ),
    concat!(
        "2024-02-01 10:00:03.000000 [ee] [ledger] [6] took=770 fill 103 SYMB-0004\n",
        "2024-02-01 10:00:04.000000 [ii] [feed] [7] took=100 fill 104 SYMB-0005\n",
    ),
];
for (index, text) in leaves.iter().enumerate() {
    std::fs::write(
        root.join(format!("app-{index}.log.gz")),
        yggdryl::gzip::dump(text.as_bytes())?,
    )?;
}

let folder = Folder::new(&root)?;
let options = TextLineOptions::with_pattern(pattern)?;
let (mut rows, mut errors, mut latency, mut traced) = (0_usize, 0_usize, 0_i64, 0_usize);

// One batch in memory at a time, each leaf decoded as a stream.
for batch in folder.read_arrow_lines(&options)? {
    let batch = batch?;
    let level = batch.column_by_name("level").unwrap();
    let level = level.as_any().downcast_ref::<StringArray>().unwrap();
    // Already an integer: nothing here parses text.
    let took = batch.column_by_name("latency_us").unwrap();
    let took = took.as_any().downcast_ref::<Int64Array>().unwrap();
    let spans = batch.column_by_name("lines").unwrap();
    let spans = spans.as_any().downcast_ref::<Int32Array>().unwrap();

    rows += batch.num_rows();
    for row in 0..batch.num_rows() {
        if spans.value(row) > 1 {
            traced += 1;
        }
        if level.value(row) == "ee" {
            errors += 1;
            latency += took.value(row);
        }
    }
}

// Five records, not the seven lines they occupy.
assert_eq!((rows, errors, traced), (5, 2, 1));
assert_eq!(latency, 980 + 770);